# 🦶 Pathological Gait & Activity Recognition

**End-to-end pipeline on real GaitRec data.**

### Steps:
1. GPU enable karo → Runtime > Change runtime type > T4 GPU
2. Har cell ko upar se neeche run karo
3. Step 2 mein zip upload hoga
4. Data automatically Figshare se download hoga
5. Classical ML + CNN dono train honge
6. Dashboard live URL milega

## Step 1 — GPU Check & Dependencies

In [ ]:
!nvidia-smi || echo 'No GPU — Runtime > Change runtime type > T4 GPU se enable karo'
!pip install -q pandas numpy scikit-learn joblib requests tqdm pyarrow streamlit plotly

## Step 2 — Zip Upload & Extract

In [ ]:
from google.colab import files
import zipfile, os, sys

print('Zip file select karo (hey-you-are-a-skilled-ai-2-fixed.zip)')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

PROJECT_ROOT = '/content/hey-you-are-a-skilled-ai-2'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f'✅ Extracted to: {PROJECT_ROOT}')

## Step 3 — GaitRec Data Download
*(Figshare collection se ~800MB — 5-10 min lagenge)*

In [ ]:
from src.gait_data import download_gaitrec, build_dataset

print('⬇️  Downloading GaitRec from Figshare...')
download_gaitrec(f'{PROJECT_ROOT}/data')
print('✅ Download complete!')

print('🔧 Building feature dataset...')
out = build_dataset(f'{PROJECT_ROOT}/data')
print(f'✅ Dataset ready: {out}')

## Step 4 — Dataset Overview

In [ ]:
import pandas as pd

df = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/gaitrec_features.parquet')
print(f'Total trials   : {len(df):,}')
print(f'Subjects       : {df["SUBJECT_ID"].nunique():,}')
print(f'Feature columns: {len([c for c in df.columns if c.startswith(("F_V","F_AP","F_ML"))]):,}')
print(f'\nClass distribution:')
print(df['target_binary'].value_counts())

## Step 5 — Classical ML Train (HistGradientBoosting)

In [ ]:
from src.train import train_baseline

print('🤖 Training Classical ML model...')
metrics_ml = train_baseline(
    data_root=f'{PROJECT_ROOT}/data',
    model_dir=f'{PROJECT_ROOT}/models',
    target='target_binary',
    max_trials=5000
)
print(f'✅ Classical ML Done!')
print(f'   Accuracy : {metrics_ml["accuracy"]:.3f}')
print(f'   Macro F1 : {metrics_ml["macro_f1"]:.3f}')

## Step 6 — 1D CNN Train (GPU)
*(GPU hone pe ~3-4 min lagenge)*

In [ ]:
from src.train_cnn import train_cnn
import torch
print(f'Device: {"GPU ✅" if torch.cuda.is_available() else "CPU ⚠️  (GPU enable karo for speed)"}')

print('🧠 Training 1D CNN...')
metrics_cnn = train_cnn(
    data_root=f'{PROJECT_ROOT}/data',
    model_dir=f'{PROJECT_ROOT}/models',
    target='target_binary',
    max_trials=25000,
    epochs=12
)
print(f'✅ CNN Done!')
print(f'   Accuracy : {metrics_cnn["accuracy"]:.3f}')
print(f'   Macro F1 : {metrics_cnn["macro_f1"]:.3f}')
print(f'   Device   : {metrics_cnn["device"]}')

## Step 7 — Model Comparison

In [ ]:
print('='*45)
print(f'{"Model":<25} {"Accuracy":>10} {"Macro F1":>10}')
print('='*45)
print(f'{"HistGradientBoosting":<25} {metrics_ml["accuracy"]:>10.3f} {metrics_ml["macro_f1"]:>10.3f}')
print(f'{"1D CNN":<25} {metrics_cnn["accuracy"]:>10.3f} {metrics_cnn["macro_f1"]:>10.3f}')
print('='*45)

## Step 8 — Live Dashboard
*(Cloudflare tunnel — no signup needed)*

In [ ]:
import subprocess, threading, time, re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

def run_streamlit():
    subprocess.run([
        'streamlit', 'run', f'{PROJECT_ROOT}/app.py',
        '--server.port', '8501',
        '--server.headless', 'true'
    ])

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(4)

tunnel = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

print('🌐 Dashboard URL dhundh raha hoon...')
for _ in range(20):
    line = tunnel.stderr.readline().decode()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        print(f'\n🚀 Dashboard URL: {match.group()}')
        print('👆 Link kholo — sidebar mein Classical ML ya CNN select karo!')
        break
    time.sleep(1)

## Step 9 — Models Download karo (optional)

In [ ]:
from google.colab import files
import os

for fname in ['gaitrec_baseline.joblib', 'gaitrec_cnn.pt',
              'gaitrec_metrics.json', 'gaitrec_cnn_metrics.json']:
    fpath = f'{PROJECT_ROOT}/models/{fname}'
    if os.path.exists(fpath):
        print(f'⬇️  Downloading {fname}...')
        files.download(fpath)
    else:
        print(f'⚠️  Not found: {fname}')